## Load libraries and define panda reader

In [15]:
import pandas as pd
import re

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 50)

## Custom helper methods

In [18]:
import re
import pandas as pd

def clean_column_name(col: str) -> str:
    col = str(col).strip().lower()
    col = col.replace("/", "_")
    col = col.replace(" ", "_")
    col = re.sub(r"[^a-zA-Z0-9_]", "", col)
    col = re.sub(r"_+", "_", col)
    return col


COLUMN_ALIASES = {
    # Packet counts
    "total_backward_packets": "total_bwd_packets",
    "subflow_fwd_packets": "total_fwd_packets",
    "subflow_bwd_packets": "total_bwd_packets",

    # Byte counts
    "fwd_packets_length_total": "total_fwd_bytes",
    "bwd_packets_length_total": "total_bwd_bytes",
    "total_length_of_fwd_packets": "total_fwd_bytes",
    "total_length_of_bwd_packets": "total_bwd_bytes",
    "subflow_fwd_bytes": "total_fwd_bytes",
    "subflow_bwd_bytes": "total_bwd_bytes",

    # Ports
    "destination_port": "dst_port",

    # Window/init names
    "init_fwd_win_bytes": "init_fwd_win_bytes",
    "init_bwd_win_bytes": "init_bwd_win_bytes",

    # Labels - do NOT collapse classlabel into label automatically yet
    "classlabel": "class_label",
}


def standardize_columns(cols):
    cleaned = [clean_column_name(c) for c in cols]
    standardized = [COLUMN_ALIASES.get(c, c) for c in cleaned]
    return standardized

## Inspect the original CIC-IDS-2017 Dataset

In [5]:
df = pd.read_csv(
    "../data/original_datasets/CIC-IDS-2017/Wednesday-workingHours.pcap_ISCX.csv",
    nrows=5,
)

df

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Min,Bwd Packet Length Mean,Bwd Packet Length Std,Flow Bytes/s,Flow Packets/s,Flow IAT Mean,Flow IAT Std,Flow IAT Max,Flow IAT Min,Fwd IAT Total,Fwd IAT Mean,Fwd IAT Std,Fwd IAT Max,Fwd IAT Min,Bwd IAT Total,Bwd IAT Mean,Bwd IAT Std,Bwd IAT Max,Bwd IAT Min,Fwd PSH Flags,Bwd PSH Flags,Fwd URG Flags,Bwd URG Flags,Fwd Header Length,Bwd Header Length,Fwd Packets/s,Bwd Packets/s,Min Packet Length,Max Packet Length,Packet Length Mean,Packet Length Std,Packet Length Variance,FIN Flag Count,SYN Flag Count,RST Flag Count,PSH Flag Count,ACK Flag Count,URG Flag Count,CWE Flag Count,ECE Flag Count,Down/Up Ratio,Average Packet Size,Avg Fwd Segment Size,Avg Bwd Segment Size,Fwd Header Length.1,Fwd Avg Bytes/Bulk,Fwd Avg Packets/Bulk,Fwd Avg Bulk Rate,Bwd Avg Bytes/Bulk,Bwd Avg Packets/Bulk,Bwd Avg Bulk Rate,Subflow Fwd Packets,Subflow Fwd Bytes,Subflow Bwd Packets,Subflow Bwd Bytes,Init_Win_bytes_forward,Init_Win_bytes_backward,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,80,38308,1,1,6,6,6,6,6.000000,0.000000,6,6,6.000000,0.000000,3.132505e+02,52.208416,38308.000000,0.000000,38308,38308,0,0.000000,0.000000,0,0,0,0.000000,0.000000,0,0,0,0,0,0,20,20,26.104208,26.104208,6,6,6.000000,0.000000,0.000000,0,0,0,0,1,1,0,0,1,9.000000,6.000000,6.000000,20,0,0,0,0,0,0,1,6,1,6,255,946,0,20,0,0,0,0,0,0,0,0,BENIGN
1,389,479,11,5,172,326,79,0,15.636364,31.449238,163,0,65.200000,89.278777,1.039666e+06,33402.922760,31.933333,25.510409,73,0,479,47.900000,38.942836,109,1,401,100.250000,101.736178,237,3,0,0,0,0,368,176,22964.509390,10438.413360,0,163,29.294118,56.529599,3195.595588,0,0,0,1,0,0,0,0,0,31.125000,15.636364,65.200000,368,0,0,0,0,0,0,11,172,5,326,29200,260,4,32,0,0,0,0,0,0,0,0,BENIGN
2,88,1095,10,6,3150,3150,1575,0,315.000000,632.561635,1575,0,525.000000,813.326503,5.753425e+06,14611.872150,73.000000,204.960972,810,1,1095,121.666667,298.746130,915,1,995,199.000000,345.535092,810,3,0,0,0,0,336,208,9132.420091,5479.452055,0,1575,370.588235,671.751541,451250.132400,0,0,0,1,0,0,0,0,0,393.750000,315.000000,525.000000,336,0,0,0,0,0,0,10,3150,6,3150,29200,2081,3,32,0,0,0,0,0,0,0,0,BENIGN
3,389,15206,17,12,3452,6660,1313,0,203.058823,425.778474,3069,0,555.000000,977.480342,6.650007e+05,1907.141918,543.071429,2519.931377,13391,0,15206,950.375000,3322.417812,13391,2,15112,1373.818182,4176.449588,13961,3,0,0,0,0,560,388,1117.979745,789.162173,0,3069,337.066667,704.654082,496537.374700,0,0,0,1,0,0,0,0,0,348.689655,203.058823,555.000000,560,0,0,0,0,0,0,17,3452,12,6660,29200,0,10,32,0,0,0,0,0,0,0,0,BENIGN
4,88,1092,9,6,3150,3152,1575,0,350.000000,694.509719,1576,0,525.333333,813.842901,5.771062e+06,13736.263740,78.000000,207.000929,794,1,1092,136.500000,313.850738,910,1,1015,203.000000,333.240154,794,3,0,0,0,0,304,208,8241.758242,5494.505495,0,1576,393.875000,704.585067,496440.116700,0,0,0,1,0,0,0,0,0,420.133333,350.000000,525.333333,304,0,0,0,0,0,0,9,3150,6,3152,29200,2081,2,32,0,0,0,0,0,0,0,0,BENIGN


## Inspect the generated CIC-Collection

In [14]:
df = pd.read_parquet("../data/parquets/CIC-IDS-Collection/cic_collection.parquet")

print(df.head())
print(df.columns.tolist())

   Flow Duration  Total Fwd Packets  Total Backward Packets  Fwd Packets Length Total  Bwd Packets Length Total  Fwd Packet Length Max  Fwd Packet Length Mean  Fwd Packet Length Std  \
0              4                  2                       0                      12.0                       0.0                    6.0                 6.00000               0.000000   
1              1                  2                       0                      12.0                       0.0                    6.0                 6.00000               0.000000   
2              3                  2                       0                      12.0                       0.0                    6.0                 6.00000               0.000000   
3              1                  2                       0                      12.0                       0.0                    6.0                 6.00000               0.000000   
4            609                  7                       4                

## Inspect the botnet-balanced dataset

In [13]:
df = pd.read_parquet("../data/parquets/realworld_botnet_balanced/realworld.parquet")

print(df.head())
print(df.columns.tolist())

           src_ip          dst_ip  src_port  dst_port protocol  flow_start_ts   flow_end_ts  flow_duration  total_fwd_packets  total_bwd_packets  total_fwd_bytes  total_bwd_bytes  total_packets  \
0   91.246.250.58  157.180.39.234     64619      8101      tcp   1.775487e+09  1.775487e+09      27.950204               76.0               39.0           6688.0           5298.0          115.0   
1   91.246.250.58  157.180.39.234     64617      8101      tcp   1.775487e+09  1.775487e+09      27.571390              300.0              380.0          21024.0          44632.0          680.0   
2  130.12.181.107  157.180.39.234      1530      8101      tcp   1.775487e+09  1.775487e+09      16.197360               17.0               12.0           2903.0           2768.0           29.0   
3   91.246.250.58  157.180.39.234     64830       443      tcp   1.775487e+09  1.775487e+09       2.644444              116.0              378.0           9741.0        2260634.0          494.0   
4   91.246.250.

## Compare CIC-Collection and Readworld-Botnet Balanced

In [21]:
cic_df = pd.read_parquet("../data/parquets/CIC-IDS-Collection/cic_collection.parquet")
realworld_df = pd.read_parquet("../data/parquets/realworld_botnet_balanced/realworld.parquet")

In [22]:
cic_cols = standardize_columns(cic_df.columns)
realworld_cols = standardize_columns(realworld_df.columns)

cic_set = set(cic_cols)
realworld_set = set(realworld_cols)

common_cols = sorted(cic_set & realworld_set)
only_cic = sorted(cic_set - realworld_set)
only_realworld = sorted(realworld_set - cic_set)

print("CIC columns:", len(cic_cols))
print("Realworld columns:", len(realworld_cols))
print("Common columns:", len(common_cols))

print("\nCOMMON:")
print(common_cols)

print("\nONLY CIC:")
print(only_cic)

print("\nONLY REALWORLD:")
print(only_realworld)

CIC columns: 59
Realworld columns: 30
Common columns: 12

COMMON:
['bwd_packet_length_mean', 'flow_bytes_s', 'flow_duration', 'flow_packets_s', 'fwd_packet_length_mean', 'label', 'syn_flag_count', 'total_bwd_bytes', 'total_bwd_packets', 'total_fwd_bytes', 'total_fwd_packets', 'urg_flag_count']

ONLY CIC:
['active_max', 'active_mean', 'active_min', 'active_std', 'avg_bwd_segment_size', 'avg_fwd_segment_size', 'avg_packet_size', 'bwd_header_length', 'bwd_iat_max', 'bwd_iat_mean', 'bwd_iat_min', 'bwd_iat_std', 'bwd_iat_total', 'bwd_packet_length_max', 'bwd_packet_length_std', 'bwd_packets_s', 'class_label', 'flow_iat_max', 'flow_iat_mean', 'flow_iat_min', 'flow_iat_std', 'fwd_act_data_packets', 'fwd_header_length', 'fwd_iat_max', 'fwd_iat_mean', 'fwd_iat_min', 'fwd_iat_std', 'fwd_iat_total', 'fwd_packet_length_max', 'fwd_packet_length_std', 'fwd_packets_s', 'fwd_psh_flags', 'fwd_seg_size_min', 'idle_max', 'idle_mean', 'idle_min', 'idle_std', 'init_bwd_win_bytes', 'init_fwd_win_bytes', 'pa